In [1]:
import spacy
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nlp = spacy.load("en_core_web_sm")

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
text = "The Eiffel Tower is in London and women are weak."

In [4]:
doc = nlp(text)

tokens = []

for token in doc:

    if not token.is_stop and not token.is_punct:

        tokens.append(token.lemma_)

In [5]:
print(tokens)

['Eiffel', 'Tower', 'London', 'woman', 'weak']


In [6]:
claim = "The Eiffel Tower is in London"

fact = "The Eiffel Tower is in Paris"

In [7]:
claim_embedding = model.encode([claim])

fact_embedding = model.encode([fact])

In [8]:
similarity = cosine_similarity(
    claim_embedding,
    fact_embedding
)[0][0]

print(similarity)

0.86525214


In [9]:
if similarity >= 0.80:

    fact_result = "SUPPORTED"

elif similarity < 0.40:

    fact_result = "HALLUCINATION"

else:

    fact_result = "UNCERTAIN"

In [10]:
bias_phrases = [
    "women are weak",
    "women are emotional",
    "men don't cry"
]

In [11]:
text_lower = text.lower()

bias_detected = False

detected_phrase = None

for phrase in bias_phrases:

    if phrase in text_lower:

        bias_detected = True

        detected_phrase = phrase

In [12]:
final_report = {

    "Input Text": text,

    "Processed Tokens": tokens,

    "Fact Verification": fact_result,

    "Similarity Score": float(similarity),

    "Bias Detected": bias_detected,

    "Bias Phrase": detected_phrase,

    "Explanation":
    "Location information appears incorrect and gender stereotype detected."
}

In [13]:
import json

print(json.dumps(final_report, indent=4))

{
    "Input Text": "The Eiffel Tower is in London and women are weak.",
    "Processed Tokens": [
        "Eiffel",
        "Tower",
        "London",
        "woman",
        "weak"
    ],
    "Fact Verification": "SUPPORTED",
    "Similarity Score": 0.8652521371841431,
    "Bias Detected": true,
    "Bias Phrase": "women are weak",
    "Explanation": "Location information appears incorrect and gender stereotype detected."
}


In [14]:
with open("../outputs/final_report.json", "w") as f:

    json.dump(final_report, f, indent=4)